# Dataset 1: medication and primary-diagnosis EDA

This split contains medication-related variables, the primary diagnosis code, and the encounter join key.

The notebook analyses this split independently. The original CSV is not changed; the cleaned copy is saved in `EDA_Dataset` for later joining on `encounter_id`.

## Setup

In [1]:
from pathlib import Path  # Import required library

import matplotlib.pyplot as plt  # Import required library
import pandas as pd  # Import required library
import seaborn as sns  # Import required library

sns.set_theme(style="whitegrid", context="notebook")  # Set the chart theme
pd.set_option("display.max_rows", 100)  # Configure dataframe display
pd.set_option("display.max_columns", 50)  # Configure dataframe display

In [2]:
def show_category_counts(series, title, top_n=15):  # Define the categorical-frequency chart helper
    counts = series.fillna("Missing").value_counts().head(top_n).sort_values()  # Calculate category frequencies
    fig, ax = plt.subplots(figsize=(10, max(3.5, 0.38 * len(counts))))  # Create the chart area
    ax.barh(counts.index.astype(str), counts.values, color="#4C78A8")  # Draw the horizontal bar chart
    ax.set_title(f"{title}: top {top_n} categories")  # Label the chart axis or title
    ax.set_xlabel("Encounter count")  # Label the chart axis or title
    ax.set_ylabel("")  # Label the chart axis or title
    for y, value in enumerate(counts.values):  # Add value labels to each bar
        ax.text(value, y, f" {value:,}", va="center", fontsize=8)  # Run this EDA or data-cleaning step
    plt.tight_layout()  # Adjust chart spacing
    plt.show()  # Display the chart


def show_numeric_distribution(series, title, discrete=False):  # Define the numeric-distribution chart helper
    values = series.dropna()  # Remove missing numeric values before plotting
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))  # Create the chart area
    sns.histplot(values, discrete=discrete, ax=axes[0], color="#4C78A8")  # Plot the feature distribution
    axes[0].set_title(f"{title}: distribution")  # Configure the numeric chart
    axes[0].set_xlabel(title)  # Configure the numeric chart
    sns.boxplot(x=values, ax=axes[1], color="#72B7B2")  # Plot the feature box plot
    axes[1].set_title(f"{title}: box plot")  # Configure the numeric chart
    axes[1].set_xlabel(title)  # Configure the numeric chart
    plt.tight_layout()  # Adjust chart spacing
    plt.show()  # Display the chart

## Load the source data

In [3]:
source_path = Path("divided_datasets") / "dataset_1.csv"  # Set the path to the source CSV
df_raw = pd.read_csv(source_path, na_values=["?"], keep_default_na=True, low_memory=False)  # Load the source CSV and convert question marks to missing values

## Initial inspection

In [4]:
df_raw.head()  # Inspect the original dataframe

,encounter_id,num_medications,metformin-pioglitazone,diabetesMed,metformin-rosiglitazone,diag_1,glimepiride,chlorpropamide,nateglinide,tolbutamide
0,2278392,1,No,No,No,250.83,No,No,No,No
1,149190,18,No,Yes,No,276,No,No,No,No
2,64410,13,No,Yes,No,648,No,No,No,No
3,500364,16,No,Yes,No,8,No,No,No,No
4,16680,8,No,Yes,No,197,No,No,No,No


In [5]:
df_raw.info()  # Inspect the original dataframe

<class 'pandas.DataFrame'>
RangeIndex: 101766 entries, 0 to 101765
Data columns (total 10 columns):
 #   Column                   Non-Null Count   Dtype
---  ------                   --------------   -----
 0   encounter_id             101766 non-null  int64
 1   num_medications          101766 non-null  int64
 2   metformin-pioglitazone   101766 non-null  str  
 3   diabetesMed              101766 non-null  str  
 4   metformin-rosiglitazone  101766 non-null  str  
 5   diag_1                   101745 non-null  str  
 6   glimepiride              101766 non-null  str  
 7   chlorpropamide           101766 non-null  str  
 8   nateglinide              101766 non-null  str  
 9   tolbutamide              101766 non-null  str  
dtypes: int64(2), str(8)
memory usage: 9.5 MB


In [6]:
df_raw.shape  # Inspect the original dataframe

(101766, 10)

In [7]:
df_raw.describe()  # Inspect the original dataframe

,encounter_id,num_medications
count,1.017660e+05,101766.000000
mean,1.652016e+08,16.021844
std,1.026403e+08,8.127566
min,1.252200e+04,1.000000
25%,8.496119e+07,10.000000
50%,1.523890e+08,15.000000
75%,2.302709e+08,20.000000
max,4.438672e+08,81.000000


In [8]:
df_raw.isna().sum().sort_values(ascending=False).to_frame(name="missing_count")  # Inspect the original dataframe

,missing_count
diag_1,21
encounter_id,0
num_medications,0
metformin-pioglitazone,0
diabetesMed,0
metformin-rosiglitazone,0
glimepiride,0
chlorpropamide,0
nateglinide,0
tolbutamide,0


In [9]:
df_raw.duplicated().sum()  # Inspect the original dataframe

np.int64(0)

In [10]:
df_raw["encounter_id"].duplicated().sum()  # Run this EDA or data-cleaning step

np.int64(0)

In [11]:
df_raw["encounter_id"].is_unique  # Run this EDA or data-cleaning step

True

In [12]:
df_raw.nunique(dropna=False).sort_values().to_frame(name="unique_values_including_missing")  # Inspect the original dataframe

,unique_values_including_missing
diabetesMed,2
metformin-pioglitazone,2
metformin-rosiglitazone,2
tolbutamide,2
chlorpropamide,4
glimepiride,4
nateglinide,4
num_medications,75
diag_1,717
encounter_id,101766


## Cleaning decisions

- Missing-value normalization: Question-mark and blank text values are treated as missing.
- Primary diagnosis: A missingness flag is retained and missing diagnosis codes are labeled Unknown for visualization.
- Rows and join key: No rows are removed. encounter_id remains unchanged and unique for the later join.

In [13]:
df = df_raw.copy()  # Copy the source dataframe before cleaning

text_columns = df.select_dtypes(include=["object", "string"]).columns  # Identify text columns for normalization
for column in text_columns:  # Clean each text column
    df[column] = df[column].astype("string").str.strip().replace("", pd.NA)  # Strip whitespace and convert blank text to missing values

df["diag_1_missing"] = df["diag_1"].isna()  # Create or handle the diagnosis missingness information
df["diag_1"] = df["diag_1"].fillna("Unknown")  # Create or handle the diagnosis missingness information
columns_removed = []  # Record columns removed during cleaning

assert len(df) == len(df_raw), "No encounter rows should be removed"  # Validate the cleaning result
assert df["encounter_id"].notna().all(), "The join key must not be missing"  # Validate the cleaning result
assert df["encounter_id"].is_unique, "The join key must remain unique"  # Validate the cleaning result
assert df.columns.is_unique, "Duplicate column names are not allowed"  # Validate the cleaning result

In [14]:
cleaning_summary = pd.DataFrame({  # Create the cleaning-decision summary
    "decision": ["Missing-value normalization","Primary diagnosis","Rows and join key","Columns removed"],  # Add a cleaning-summary field
    "detail": ["Question-mark and blank text values are treated as missing.","A missingness flag is retained and missing diagnosis codes are labeled Unknown for visualization.","No rows are removed. encounter_id remains unchanged and unique for the later join.","Columns removed after review: none"]  # Add a cleaning-summary field
})  # Run this EDA or data-cleaning step
cleaning_summary  # Create the cleaning-decision summary

,decision,detail
0,Missing-value normalization,Question-mark and blank text values are treate...
1,Primary diagnosis,A missingness flag is retained and missing dia...
2,Rows and join key,No rows are removed. encounter_id remains unch...
3,Columns removed,Columns removed after review: none


## Post-cleaning validation

In [15]:
df.head()  # Run this EDA or data-cleaning step

,encounter_id,num_medications,metformin-pioglitazone,diabetesMed,metformin-rosiglitazone,diag_1,glimepiride,chlorpropamide,nateglinide,tolbutamide,diag_1_missing
0,2278392,1,No,No,No,250.83,No,No,No,No,False
1,149190,18,No,Yes,No,276,No,No,No,No,False
2,64410,13,No,Yes,No,648,No,No,No,No,False
3,500364,16,No,Yes,No,8,No,No,No,No,False
4,16680,8,No,Yes,No,197,No,No,No,No,False


In [16]:
df.info()  # Run this EDA or data-cleaning step

<class 'pandas.DataFrame'>
RangeIndex: 101766 entries, 0 to 101765
Data columns (total 11 columns):
 #   Column                   Non-Null Count   Dtype 
---  ------                   --------------   ----- 
 0   encounter_id             101766 non-null  int64 
 1   num_medications          101766 non-null  int64 
 2   metformin-pioglitazone   101766 non-null  string
 3   diabetesMed              101766 non-null  string
 4   metformin-rosiglitazone  101766 non-null  string
 5   diag_1                   101766 non-null  string
 6   glimepiride              101766 non-null  string
 7   chlorpropamide           101766 non-null  string
 8   nateglinide              101766 non-null  string
 9   tolbutamide              101766 non-null  string
 10  diag_1_missing           101766 non-null  bool  
dtypes: bool(1), int64(2), string(8)
memory usage: 9.7 MB


In [17]:
df.shape  # Run this EDA or data-cleaning step

(101766, 11)

In [18]:
df.describe()  # Run this EDA or data-cleaning step

,encounter_id,num_medications
count,1.017660e+05,101766.000000
mean,1.652016e+08,16.021844
std,1.026403e+08,8.127566
min,1.252200e+04,1.000000
25%,8.496119e+07,10.000000
50%,1.523890e+08,15.000000
75%,2.302709e+08,20.000000
max,4.438672e+08,81.000000


In [19]:
df.isna().sum().sort_values(ascending=False).to_frame(name="missing_count_after_cleaning")  # Run this EDA or data-cleaning step

,missing_count_after_cleaning
encounter_id,0
num_medications,0
metformin-pioglitazone,0
diabetesMed,0
metformin-rosiglitazone,0
diag_1,0
glimepiride,0
chlorpropamide,0
nateglinide,0
tolbutamide,0


In [20]:
df["encounter_id"].nunique()  # Inspect the selected cleaned column

101766

## Univariate EDA after cleaning

### Number of medications

In [21]:
df["num_medications"].describe()  # Inspect the selected cleaned column

count    101766.000000
mean         16.021844
std           8.127566
min           1.000000
25%          10.000000
50%          15.000000
75%          20.000000
max          81.000000
Name: num_medications, dtype: float64

In [22]:
show_numeric_distribution(df["num_medications"], "Number of medications", discrete=True)  # Run this EDA or data-cleaning step

C:\Users\anind\AppData\Local\Temp\ipykernel_14100\4155684497.py:24: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()  # Display the chart


### Primary diagnosis code

In [23]:
df["diag_1"].value_counts(dropna=False).head(20).to_frame(name="count")  # Create or handle the diagnosis missingness information

,count
diag_1,
428,6862
414,6581
786,4016
410,3614
486,3508
427,2766
491,2275
715,2151
682,2042


In [24]:
show_category_counts(df["diag_1"], "Primary diagnosis code", top_n=20)  # Run this EDA or data-cleaning step

C:\Users\anind\AppData\Local\Temp\ipykernel_14100\4155684497.py:11: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()  # Display the chart


### Metformin-pioglitazone status

In [25]:
df["metformin-pioglitazone"].value_counts(dropna=False).head(10).to_frame(name="count")  # Inspect the selected cleaned column

,count
metformin-pioglitazone,
No,101765
Steady,1


In [26]:
show_category_counts(df["metformin-pioglitazone"], "Metformin-pioglitazone status", top_n=10)  # Run this EDA or data-cleaning step

C:\Users\anind\AppData\Local\Temp\ipykernel_14100\4155684497.py:11: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()  # Display the chart


### Diabetes medication prescribed

In [27]:
df["diabetesMed"].value_counts(dropna=False).head(10).to_frame(name="count")  # Inspect the selected cleaned column

,count
diabetesMed,
Yes,78363
No,23403


In [28]:
show_category_counts(df["diabetesMed"], "Diabetes medication prescribed", top_n=10)  # Run this EDA or data-cleaning step

C:\Users\anind\AppData\Local\Temp\ipykernel_14100\4155684497.py:11: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()  # Display the chart


### Metformin-rosiglitazone status

In [29]:
df["metformin-rosiglitazone"].value_counts(dropna=False).head(10).to_frame(name="count")  # Inspect the selected cleaned column

,count
metformin-rosiglitazone,
No,101764
Steady,2


In [30]:
show_category_counts(df["metformin-rosiglitazone"], "Metformin-rosiglitazone status", top_n=10)  # Run this EDA or data-cleaning step

C:\Users\anind\AppData\Local\Temp\ipykernel_14100\4155684497.py:11: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()  # Display the chart


### Glimepiride status

In [31]:
df["glimepiride"].value_counts(dropna=False).head(10).to_frame(name="count")  # Inspect the selected cleaned column

,count
glimepiride,
No,96575
Steady,4670
Up,327
Down,194


In [32]:
show_category_counts(df["glimepiride"], "Glimepiride status", top_n=10)  # Run this EDA or data-cleaning step

C:\Users\anind\AppData\Local\Temp\ipykernel_14100\4155684497.py:11: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()  # Display the chart


### Chlorpropamide status

In [33]:
df["chlorpropamide"].value_counts(dropna=False).head(10).to_frame(name="count")  # Inspect the selected cleaned column

,count
chlorpropamide,
No,101680
Steady,79
Up,6
Down,1


In [34]:
show_category_counts(df["chlorpropamide"], "Chlorpropamide status", top_n=10)  # Run this EDA or data-cleaning step

C:\Users\anind\AppData\Local\Temp\ipykernel_14100\4155684497.py:11: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()  # Display the chart


### Nateglinide status

In [35]:
df["nateglinide"].value_counts(dropna=False).head(10).to_frame(name="count")  # Inspect the selected cleaned column

,count
nateglinide,
No,101063
Steady,668
Up,24
Down,11


In [36]:
show_category_counts(df["nateglinide"], "Nateglinide status", top_n=10)  # Run this EDA or data-cleaning step

C:\Users\anind\AppData\Local\Temp\ipykernel_14100\4155684497.py:11: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()  # Display the chart


### Tolbutamide status

In [37]:
df["tolbutamide"].value_counts(dropna=False).head(10).to_frame(name="count")  # Inspect the selected cleaned column

,count
tolbutamide,
No,101743
Steady,23


In [38]:
show_category_counts(df["tolbutamide"], "Tolbutamide status", top_n=10)  # Run this EDA or data-cleaning step

C:\Users\anind\AppData\Local\Temp\ipykernel_14100\4155684497.py:11: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()  # Display the chart


## Save the cleaned split

In [39]:
output_dir = Path("EDA_Dataset")  # Set the cleaned-data output folder
output_dir.mkdir(exist_ok=True)  # Set the cleaned-data output folder
output_path = output_dir / "dataset_1.csv"  # Set the cleaned CSV output path
df.to_csv(output_path, index=False)  # Save the cleaned dataframe
print(f"Saved {df.shape[0]:,} rows and {df.shape[1]:,} columns to {output_path}")  # Report the saved dataset details

Saved 101,766 rows and 11 columns to EDA_Dataset\dataset_1.csv


In [40]:
saved_df = pd.read_csv(output_path)  # Reload the saved CSV for verification
assert saved_df.shape == df.shape, "Saved CSV shape does not match the cleaned dataframe"  # Validate the cleaning result
saved_df.head()  # Reload the saved CSV for verification

,encounter_id,num_medications,metformin-pioglitazone,diabetesMed,metformin-rosiglitazone,diag_1,glimepiride,chlorpropamide,nateglinide,tolbutamide,diag_1_missing
0,2278392,1,No,No,No,250.83,No,No,No,No,False
1,149190,18,No,Yes,No,276,No,No,No,No,False
2,64410,13,No,Yes,No,648,No,No,No,No,False
3,500364,16,No,Yes,No,8,No,No,No,No,False
4,16680,8,No,Yes,No,197,No,No,No,No,False
